# CoCoA-CoT Experiments on Google Colab

This notebook runs the complete CoCoA-CoT experimental pipeline on Google Colab with GPU support.

**Features:**
- Full environment setup with GPU detection
- Automatic repository cloning and package installation
- Configurable experiment parameters
- All 6 experiment stages (Main, Ablation, Alpha, Calibration, Black-box, Light)
- Results download capability

**GPU Requirements:** T4 or higher recommended (A100 for faster training)

## 1. Setup and Dependencies

Install required Python packages for the CoCoA-CoT pipeline.

In [1]:
# Check GPU availability
import subprocess
import sys

gpu_result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print("GPU Status:")
print(gpu_result.stdout if gpu_result.returncode == 0 else "No GPU detected")

# Install system dependencies
print("\n" + "="*60)
print("Installing system dependencies...")
print("="*60)
subprocess.check_call(['apt-get', 'update'])
subprocess.check_call(['apt-get', 'install', '-y', 'git'])

# Install Python packages
print("\n" + "="*60)
print("Installing Python dependencies...")
print("="*60)

packages = [
    'torch>=2.0.0',
    'transformers>=4.35.0',
    'peft>=0.4.0',
    'numpy',
    'pandas',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'pyyaml',
    'tqdm',
    'datasets',
    'sentence-transformers',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print("✓ All dependencies installed successfully!")


GPU Status:
Sat Mar 14 21:13:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------

## 2. Configure Local Output Directories

Set up local directories for experiment results.

In [3]:
import os

# Configure local paths
WORK_DIR = '/content/cocoa_cot'  # or use '/tmp/cocoa_cot' if you prefer /tmp
RESULTS_DIR = '/tmp/cocoa_cot_results'

# Create output directories
os.makedirs(f'{RESULTS_DIR}/tables', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/figures', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/cache', exist_ok=True)

print(f"✓ Working directory: {WORK_DIR}")
print(f"✓ Results directory: {RESULTS_DIR}")
print(f"✓ Output folders created successfully!")

✓ Working directory: /content/cocoa_cot
✓ Results directory: /tmp/cocoa_cot_results
✓ Output folders created successfully!


## 3. Clone Repository and Install Package

Clone the CoCoA-CoT repository and install in development mode.

In [16]:
import subprocess
import os
import sys

# Clone repository (or skip if already exists)
if os.path.exists(WORK_DIR):
    print("Cloning CoCoA-CoT repository...")
    subprocess.check_call([
        'git', 'clone',
        'https://github.com/vlntlbr/cocoa_cot_v2.git',  # Update with correct repo URL
        WORK_DIR
    ])
# else:
#     print("Repository already exists, skipping clone.")

# Change to working directory
os.chdir(WORK_DIR)

# Install package in development mode
print("\nInstalling CoCoA-CoT package in development mode...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'])

# Download required models (optional, comment out if running short experiments)
print("\nDownloading required models...")
subprocess.run(['bash', 'scripts/download_models.sh'], cwd=WORK_DIR)

print("✓ Repository cloned and package installed successfully!")
print(f"✓ Working directory: {os.getcwd()}")


Cloning CoCoA-CoT repository...


CalledProcessError: Command '['git', 'clone', 'https://github.com/vlntlbr/cocoa_cot_v2.git', '/content/cocoa_cot']' returned non-zero exit status 128.

## 4. Configure Experiment Parameters

Define experiment hyperparameters. Adjust these values based on your needs and available memory.

In [5]:
# Experiment Configuration
CONFIG = {
    # Model settings
    'model': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',  # Options: gpt2, mistral, llama, deepseek, etc.
    'config_path': 'configs/base.yaml',
    
    # Evaluation settings
    'n_eval': 100,  # Number of eval samples per dataset. Use 10-50 for quick test, 100-500 for full
    'n_holdout': 500,  # Number of holdout samples for calibration/light training (use 2000 for full)
    
    # Random seeds
    'seeds': [42],  # Add more seeds for robustness: [42, 123, 456]
    
    # Output settings
    'output_dir': RESULTS_DIR,
}

# Print configuration
print("Experiment Configuration:")
print("-" * 60)
for key, value in CONFIG.items():
    print(f"  {key:20s}: {value}")
print("-" * 60)

# Verify config file exists
import os
config_file = os.path.join(WORK_DIR, CONFIG['config_path'])
if os.path.exists(config_file):
    print(f"✓ Config file found: {config_file}")
else:
    print(f"✗ Config file not found: {config_file}")


Experiment Configuration:
------------------------------------------------------------
  model               : deepseek-ai/DeepSeek-R1-Distill-Llama-8B
  config_path         : configs/base.yaml
  n_eval              : 100
  n_holdout           : 500
  seeds               : [42]
  output_dir          : /tmp/cocoa_cot_results
------------------------------------------------------------
✓ Config file found: /content/cocoa_cot/configs/base.yaml


## 5. Run Main Results (Table 1)

Execute the main results experiment across all datasets and methods.

**Datasets:** GSM8K, ARC-Challenge, HotpotQA, MATH500, ProofWriter, LiveCodeBench

⏱️ **Estimated time:** 20-60 minutes depending on n_eval and model size

In [ ]:
import subprocess
import os
from datetime import datetime

os.chdir(WORK_DIR)

print("="*70)
print("[STEP 1/6] Main Results — Table 1")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

output_file = os.path.join(CONFIG['output_dir'], 'tables', 'main_results.csv')

try:
    cmd = [
        'python', '-m', 'cocoa_cot.experiments.run_main',
        '--config', CONFIG['config_path'],
        '--model', CONFIG['model'],
        '--n-eval', str(1),
        '--output', output_file,
    ]
    
    print(f"Command: {' '.join(cmd)}\n")
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    print()
    print(f"✓ Main results saved to: {output_file}")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except subprocess.CalledProcessError as e:
    print(f"✗ Error running main results (exit code {e.returncode})")
    print("\nSTDOUT:")
    print(e.stdout)
    print("\nSTDERR:")
    print(e.stderr)
    raise

[STEP 1/6] Main Results — Table 1
Start time: 2026-03-14 21:24:10

Command: python -m cocoa_cot.experiments.run_main --config configs/base.yaml --model deepseek-ai/DeepSeek-R1-Distill-Llama-8B --n-eval 2 --output /tmp/cocoa_cot_results/tables/main_results.csv

✗ Error running main results: Command '['python', '-m', 'cocoa_cot.experiments.run_main', '--config', 'configs/base.yaml', '--model', 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', '--n-eval', '2', '--output', '/tmp/cocoa_cot_results/tables/main_results.csv']' returned non-zero exit status 1.


CalledProcessError: Command '['python', '-m', 'cocoa_cot.experiments.run_main', '--config', 'configs/base.yaml', '--model', 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', '--n-eval', '2', '--output', '/tmp/cocoa_cot_results/tables/main_results.csv']' returned non-zero exit status 1.

## 6. Run Ablation Study (Table 2)

Run the ablation study to analyze component contributions.

**Components tested:** Chain parsing, step segmentation, uncertainty estimation

⏱️ **Estimated time:** 20-60 minutes

In [ ]:
import subprocess
import os
from datetime import datetime

os.chdir(WORK_DIR)

print("="*70)
print("[STEP 2/6] Ablation Study — Table 2")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

output_file = os.path.join(CONFIG['output_dir'], 'tables', 'ablation.csv')

try:
    cmd = [
        'python', '-m', 'cocoa_cot.experiments.run_ablation',
        '--config', CONFIG['config_path'],
        '--model', CONFIG['model'],
        '--n-eval', str(CONFIG['n_eval']),
        '--output', output_file,
    ]
    
    print(f"Command: {' '.join(cmd)}\n")
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    print()
    print(f"✓ Ablation results saved to: {output_file}")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except subprocess.CalledProcessError as e:
    print(f"✗ Error running ablation study (exit code {e.returncode})")
    print("\nSTDOUT:")
    print(e.stdout)
    print("\nSTDERR:")
    print(e.stderr)
    raise

## 7. Run Alpha Sensitivity Analysis (Table 3)

Execute alpha sensitivity analysis and generate heatmap visualizations.

**Purpose:** Analyze sensitivity to the alpha hyperparameter across datasets

⏱️ **Estimated time:** 20-60 minutes

In [ ]:
import subprocess
import os
from datetime import datetime

os.chdir(WORK_DIR)

print("="*70)
print("[STEP 3/6] Alpha Sensitivity Analysis — Table 3")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

output_file = os.path.join(CONFIG['output_dir'], 'tables', 'alpha_sensitivity.csv')
figure_file = os.path.join(CONFIG['output_dir'], 'figures', 'alpha_heatmap.png')

try:
    cmd = [
        'python', '-m', 'cocoa_cot.experiments.run_alpha',
        '--config', CONFIG['config_path'],
        '--model', CONFIG['model'],
        '--n-eval', str(CONFIG['n_eval']),
        '--output', output_file,
        '--figure-path', figure_file,
    ]
    
    print(f"Command: {' '.join(cmd)}\n")
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    print()
    print(f"✓ Alpha sensitivity results saved to: {output_file}")
    print(f"✓ Heatmap figure saved to: {figure_file}")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except subprocess.CalledProcessError as e:
    print(f"✗ Error running alpha sensitivity analysis (exit code {e.returncode})")
    print("\nSTDOUT:")
    print(e.stdout)
    print("\nSTDERR:")
    print(e.stderr)
    raise

## 8. Run Calibration Analysis

Perform calibration analysis with reliability diagrams and save results.

**Purpose:** Measure confidence calibration and generate reliability diagrams

⏱️ **Estimated time:** 20-60 minutes

In [ ]:
import subprocess
import os
from datetime import datetime

os.chdir(WORK_DIR)

print("="*70)
print("[STEP 4/6] Calibration Analysis")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

output_file = os.path.join(CONFIG['output_dir'], 'tables', 'calibration.csv')
figure_dir = os.path.join(CONFIG['output_dir'], 'figures', 'calibration')
os.makedirs(figure_dir, exist_ok=True)

try:
    cmd = [
        'python', '-m', 'cocoa_cot.experiments.run_calibration',
        '--config', CONFIG['config_path'],
        '--model', CONFIG['model'],
        '--n-eval', str(CONFIG['n_eval']),
        '--n-holdout', str(CONFIG['n_holdout']),
        '--output', output_file,
        '--figure-dir', figure_dir,
    ]
    
    print(f"Command: {' '.join(cmd)}\n")
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    print()
    print(f"✓ Calibration results saved to: {output_file}")
    print(f"✓ Reliability diagrams saved to: {figure_dir}")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except subprocess.CalledProcessError as e:
    print(f"✗ Error running calibration analysis (exit code {e.returncode})")
    print("\nSTDOUT:")
    print(e.stdout)
    print("\nSTDERR:")
    print(e.stderr)
    raise

## 9. Run Black-box Experiment

Execute the black-box experiment to test model-agnostic uncertainty estimation.

⏱️ **Estimated time:** 20-60 minutes

In [ ]:
import subprocess
import os
from datetime import datetime

os.chdir(WORK_DIR)

print("="*70)
print("[STEP 5/6] Black-box Experiment")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

output_file = os.path.join(CONFIG['output_dir'], 'tables', 'blackbox.csv')

try:
    cmd = [
        'python', '-m', 'cocoa_cot.experiments.run_blackbox',
        '--config', CONFIG['config_path'],
        '--model', CONFIG['model'],
        '--n-eval', str(CONFIG['n_eval']),
        '--output', output_file,
    ]
    
    print(f"Command: {' '.join(cmd)}\n")
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    print()
    print(f"✓ Black-box results saved to: {output_file}")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except subprocess.CalledProcessError as e:
    print(f"✗ Error running black-box experiment (exit code {e.returncode})")
    print("\nSTDOUT:")
    print(e.stdout)
    print("\nSTDERR:")
    print(e.stderr)
    raise

## 10. Run CoCoA-CoT Light

Train and evaluate the lightweight CoCoA-CoT model for efficient deployment.

**Purpose:** Demonstrate efficient uncertainty estimation with minimal computational overhead

⏱️ **Estimated time:** 30-90 minutes (includes training)

In [ ]:
import subprocess
import os
from datetime import datetime

os.chdir(WORK_DIR)

print("="*70)
print("[STEP 6/6] CoCoA-CoT Light")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

output_file = os.path.join(CONFIG['output_dir'], 'tables', 'light.csv')
model_save_path = os.path.join(CONFIG['output_dir'], 'cache', 'aux_model.pt')
figure_dir = os.path.join(CONFIG['output_dir'], 'figures', 'light')
os.makedirs(figure_dir, exist_ok=True)
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

try:
    cmd = [
        'python', '-m', 'cocoa_cot.experiments.run_light',
        '--config', CONFIG['config_path'],
        '--model', CONFIG['model'],
        '--n-eval', str(CONFIG['n_eval']),
        '--n-holdout', str(CONFIG['n_holdout']),
        '--output', output_file,
        '--model-save-path', model_save_path,
        '--figure-dir', figure_dir,
    ]
    
    print(f"Command: {' '.join(cmd)}\n")
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    print()
    print(f"✓ Light model results saved to: {output_file}")
    print(f"✓ Model weights saved to: {model_save_path}")
    print(f"✓ Figures saved to: {figure_dir}")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except subprocess.CalledProcessError as e:
    print(f"✗ Error running CoCoA-CoT Light (exit code {e.returncode})")
    print("\nSTDOUT:")
    print(e.stdout)
    print("\nSTDERR:")
    print(e.stderr)
    raise

## 11. Results Summary

Print and display all experimental outputs.

In [ ]:
import os
from datetime import datetime

print("="*70)
print("Results Summary")
print("="*70)

results_dir = os.path.join(CONFIG['output_dir'])

# List all output files
print("\nGenerated Files:")
print("-" * 70)

if os.path.exists(os.path.join(results_dir, 'tables')):
    tables = sorted(os.listdir(os.path.join(results_dir, 'tables')))
    if tables:
        print("\nTables:")
        for f in tables:
            fpath = os.path.join(results_dir, 'tables', f)
            size = os.path.getsize(fpath) / 1024  # KB
            print(f"  ✓ {f:30s} ({size:8.1f} KB)")

if os.path.exists(os.path.join(results_dir, 'figures')):
    print("\nFigures:")
    for root, dirs, files in os.walk(os.path.join(results_dir, 'figures')):
        for f in sorted(files):
            fpath = os.path.join(root, f)
            size = os.path.getsize(fpath) / 1024  # KB
            rel_path = os.path.relpath(fpath, results_dir)
            print(f"  ✓ {rel_path:40s} ({size:8.1f} KB)")

if os.path.exists(os.path.join(results_dir, 'cache')):
    cache_files = [f for f in os.listdir(os.path.join(results_dir, 'cache')) if f.endswith('.pt')]
    if cache_files:
        print("\nModel Checkpoints:")
        for f in sorted(cache_files):
            fpath = os.path.join(results_dir, 'cache', f)
            size = os.path.getsize(fpath) / (1024 * 1024)  # MB
            print(f"  ✓ {f:30s} ({size:8.1f} MB)")

print("\n" + "="*70)
print("All results are stored locally at:")
print(f"  {RESULTS_DIR}")
print("="*70)
print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Appendix A: Quick Test (Optional)

Run a minimal test to verify everything works without full experiments.

In [ ]:
# Quick test on GSM8K with minimal eval samples
import subprocess
import os

os.chdir(WORK_DIR)

print("="*70)
print("Quick Test: Running main experiments on GSM8K (n_eval=5)")
print("="*70)
print("This is a minimal test to verify the setup works.\n")

try:
    cmd = [
        'python', '-m', 'cocoa_cot.experiments.run_main',
        '--config', CONFIG['config_path'],
        '--model', CONFIG['model'],
        '--n-eval', '5',  # Minimal samples for quick test
        '--output', '/tmp/quick_test.csv',
    ]
    
    print(f"Command: {' '.join(cmd)}\n")
    subprocess.run(cmd, check=True, timeout=600)  # 10 minute timeout
    
    print("\n✓ Quick test completed successfully!")
    print("  All systems operational. Ready to run full experiments.")
    
except subprocess.TimeoutExpired:
    print("\n✗ Test timed out (10 minutes). Check GPU memory or try fewer samples.")
except subprocess.CalledProcessError as e:
    print(f"\n✗ Test failed: {e}")
    print("  Review error messages above for troubleshooting.")
